Buscar criar um histórico de linhas para cada veículo, ver se segue algum padrão ou é totalmente randômico.

In [1]:
import pandas as pd

In [2]:
veiculos = pd.read_csv('dados_geral/veiculo/veiculos_em_linha.csv')
registros_viagem = pd.read_csv('dados_geral/registros_viagem/viagem_informada.csv')

In [ ]:
print(veiculos.info())
print(registros_viagem.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6598 entries, 0 to 6597
Data columns (total 29 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   data                            6598 non-null   object 
 1   modo                            6598 non-null   object 
 2   id_veiculo                      6598 non-null   object 
 3   ano_fabricacao                  6598 non-null   int64  
 4   carroceria                      6598 non-null   object 
 5   data_ultima_vistoria            5854 non-null   object 
 6   id_carroceria                   6598 non-null   int64  
 7   id_chassi                       6598 non-null   int64  
 8   id_fabricante_chassi            6598 non-null   int64  
 9   id_interno_carroceria           6598 non-null   int64  
 10  id_planta                       6598 non-null   int64  
 11  indicador_ar_condicionado       6598 non-null   bool   
 12  indicador_elevador              65

In [23]:
#ids como conjunto
ids_em_linha = set(veiculos["id_veiculo"].unique())
ids_registros_viagem = set(registros_viagem["id_veiculo"].unique())

#ids que estão em registros mas fora de linha
fora_de_linha = ids_registros_viagem - ids_em_linha
nao_usados = ids_em_linha - ids_registros_viagem

if fora_de_linha:
    print(f"Há {len(fora_de_linha)} IDs 'fora de linha' presentes nos registros.")
    print(f"Exemplos: {list(fora_de_linha)[:5]}")
    print(len(ids_registros_viagem))
    print(len(ids_em_linha))
if nao_usados:
    print(f"Há {len(nao_usados)} IDs 'não usados' presentes em linha.")
    print(f"Exemplos: {list(nao_usados)[:5]}")
entradas_fora_de_linha = registros_viagem[registros_viagem["id_veiculo"].isin(fora_de_linha)]
print(entradas_fora_de_linha.shape)


Há 501 IDs 'fora de linha' presentes nos registros.
Exemplos: ['D33188', 'D33107', 'D33330', 'B58115-2', 'D33294']
4863
6598
Há 2236 IDs 'não usados' presentes em linha.
Exemplos: ['B10505', 'B25610', 'D86702', 'M902247', 'M902177']
(7018, 14)


A solução foi simplificada. Estamos usando apenas a interseção para fazer o históricos. Não vamos lidar com esses dados. Além disso, para especializar nossa abordagem na questão do ar condicionado, vale simplificar a tabela de veículos em linha. O resto, da tabela de registros de viagem vai ser mantido.

In [20]:
#"Veiculos" simplificado
veiculos_simplificado = veiculos.loc[:,['id_veiculo','indicador_ar_condicionado']]

In [25]:
#Join das tabelas
veiculos_em_linha_em_viagem = veiculos_simplificado.merge(
    registros_viagem, 
    on='id_veiculo', 
    how='inner',
    suffixes=('_veic', '_viag')
)
print(veiculos_em_linha_em_viagem.info())
print(veiculos_em_linha_em_viagem["route_id"].nunique())
#Teste: print((veiculos_em_linha_em_viagem.shape[0]+entradas_fora_de_linha.shape[0])==registros_viagem.shape[0])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249707 entries, 0 to 249706
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id_veiculo                   249707 non-null  object 
 1   indicador_ar_condicionado    249707 non-null  bool   
 2   data                         249707 non-null  object 
 3   datetime_partida             249707 non-null  object 
 4   datetime_chegada             249707 non-null  object 
 5   datetime_processamento       249707 non-null  object 
 6   datetime_captura             249707 non-null  object 
 7   trip_id                      0 non-null       float64
 8   route_id                     248866 non-null  object 
 9   shape_id                     248021 non-null  object 
 10  servico                      249707 non-null  object 
 11  sentido                      249707 non-null  object 
 12  id_viagem                    249707 non-null  object 
 13 

Agora tenho todos os veículos por cada trip realizada. Fazer agora um map id->route. se houver alguma route muito relevante, eu travo id->rota e analiso essa rota e o ar condicionado. se não, excluo novamente

In [45]:
#Par veiculo - rotas
contagem_rotas = veiculos_em_linha_em_viagem.groupby(['id_veiculo', 'route_id']).size().reset_index(name='frequencia')
#Frquência relativa de cada rota em relação ao total de trips do veículo
contagem_rotas["frequencia_relativa"] = contagem_rotas['frequencia'] / contagem_rotas.groupby('id_veiculo')['frequencia'].transform('sum')
# Filtragem rigorosa: mantém apenas registros com proporção > 0.5
veiculos_dominantes = contagem_rotas[contagem_rotas['frequencia_relativa'] > 0.5]

# Ordenação para análise (opcional)
veiculos_dominantes = veiculos_dominantes.sort_values(by='frequencia_relativa', ascending=False)

print(veiculos_dominantes.shape)
print(contagem_rotas.groupby('id_veiculo').size().shape)

(2167, 4)
(4362,)


Dada a conclusão, associar o veículo à uma rota, ou não, e usar os dados de shape que existem para trabalhar em cima dos dados da onda de calor.
Penso em enriquecer com gráficos para facilitar a visualização, e tomar a decisão direito.